# TP 5 — Jointures, audit référentiel et entonnoir de conversion

**Big Data Engineering — Master 1 — ISI — Prof. Samba Ndiaye**
etudiant : René legrand mountata


Séance 5 — Spark SQL avancé et analyse métier.

**Consignes :**
1. Complétez toutes les cellules marquées `# === À COMPLÉTER ===` (les `...` indiquent les trous) ;
2. Remplissez le **tableau de relevés** au fil des exercices ;
3. Rédigez les réponses aux questions de réflexion (cellules « *Votre réponse :* ») ;
4. Exécutez le notebook **de bout en bout** (« Restart & Run All ») avant de le pousser **avec ses sorties**.

**Livrable :** ce notebook, dans `notebooks/` de votre dépôt GitHub, poussé **avant la séance 6**.


## 0. Vérification de l'environnement

Comme aux TP précédents : Python ≥ 3.9, PySpark installé, données du fil rouge
générées à l'**échelle 0.1** avec la **graine 42** (par défaut du script).
Si le dossier `data/` est absent (ou après un reset Colab), dé-commentez la
cellule de génération.

In [33]:
import sys, platform
print("Python :", sys.version.split()[0], "-", platform.system())

import pyspark
print("PySpark :", pyspark.__version__)

Python : 3.12.0 - Windows
PySpark : 3.5.9


In [34]:
# Si necessaire (environ 1 minute a l'echelle 0.1) :
# !python3 generate_data.py --scale 0.1 --outdir ../data

import os
attendus = ["customers.csv", "products.csv", "orders.csv",
            "order_items.csv", "payments.json", "events.json"]
# BUG corrige : le chemin utilise ici ("data/") ne correspondait pas au
# chemin reellement utilise pour charger les fichiers plus bas ("../data/"),
# ce qui donnait un faux-positif "fichiers manquants".
manquants = [f for f in attendus if not os.path.exists(os.path.join("../data", f))]
print("Fichiers manquants :", manquants if manquants else "aucun - OK")


Fichiers manquants : aucun - OK


In [35]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("TP5_jointures")
         .master("local[*]")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "- session prete")

Spark 3.5.9 - session prete


## Tableau de relevés

Remplissez ce dictionnaire **au fil du TP** (ré-exécutez la cellule après
chaque mise à jour). Graine 42 oblige : vos valeurs doivent être identiques
à celles de vos voisins — comparez-les, c'est un contrôle gratuit.

In [36]:
releves = {
    "R1_count_commandes_clean":      None,  # partie A
    "R2_count_apres_jointure":       None,  # partie B1
    "R3_ca_sentinelle_avant_apres":  None,  # partie B3 (tuple)
    "R4_orphelines_count_et_part":   None,  # partie C1 (tuple)
    "R5_ca_orphelines":              None,  # partie C2
    "R6_entonnoir_sessions":         None,  # partie D1 (tuple de 4)
    "R7_taux_etape_a_etape":         None,  # partie D1 (tuple de 3, en %)
    "R8_purchase_orphelins":         None,  # partie D3
}
releves

{'R1_count_commandes_clean': None,
 'R2_count_apres_jointure': None,
 'R3_ca_sentinelle_avant_apres': None,
 'R4_orphelines_count_et_part': None,
 'R5_ca_orphelines': None,
 'R6_entonnoir_sessions': None,
 'R7_taux_etape_a_etape': None,
 'R8_purchase_orphelins': None}

## Partie A — Mise en place et reconstruction de `commandes_clean` (20 min)

On recharge les quatre tables « transactionnelles » et on reconstruit la vue
nettoyée du TP 4. Rappel du piège : ~1 % des montants de `orders.csv` portent
un suffixe « FCFA », ce qui force **toute la colonne** en `string`.

In [38]:
orders = spark.read.option("header", True).csv("../data/orders.csv")
customers = spark.read.option("header", True).csv("../data/customers.csv")
products = spark.read.option("header", True).csv("../data/products.csv")
items = spark.read.option("header", True).csv("../data/order_items.csv")

orders.createOrReplaceTempView("commandes_brutes")
customers.createOrReplaceTempView("clients")
products.createOrReplaceTempView("produits")
items.createOrReplaceTempView("lignes_commande")

print("orders        :", orders.count())
print("customers     :", customers.count())
print("products      :", products.count())
print("order_items   :", items.count())

orders        : 500000
customers     : 50250
products      : 2000
order_items   : 1129775


### A1 — La vue `commandes_clean`

Recréez la vue du TP 4 : montant nettoyé (suppression de tout caractère non
numérique) puis casté en `BIGINT`.

In [39]:
# === À COMPLÉTER ===
spark.sql("""
CREATE OR REPLACE TEMP VIEW commandes_clean AS
SELECT order_id, customer_id, date_commande, statut, canal,
       CAST(regexp_replace(montant_total_fcfa, '[^0-9]', '') AS BIGINT) AS montant_total_fcfa
FROM commandes_brutes
""")

total_commandes = spark.table("commandes_clean").count()
releves["R1_count_commandes_clean"] = total_commandes
print("Releve R1 :", total_commandes)

Releve R1 : 500000


## Partie B — Jointures multi-tables contrôlées (35 min)

### B1 — Le CA par région, avec la discipline du cours

**Comptage avant, jointure, comptage après** — puis une phrase d'explication.

In [40]:
# === À COMPLÉTER ===
spark.sql("""
CREATE OR REPLACE TEMP VIEW ventes_regions AS
SELECT o.*, c.ville, c.region
FROM   commandes_clean o
JOIN    clients c ON o.customer_id = c.customer_id
""")

apres = spark.table("ventes_regions").count()
releves["R2_count_apres_jointure"] = apres
print("Avant :", total_commandes, "| Apres :", apres)

Avant : 500000 | Apres : 502111


**Question B1 :** le comptage après jointure est-il égal au relevé R1 ?
Expliquez la différence en une ou deux phrases (vous vérifierez votre
hypothèse en partie C).

*Votre réponse :* Non : R2 (502 111) est **supérieur** à R1 (500 000), soit
2 111 lignes de plus après la jointure. Le nombre de commandes n'a pas
diminué (donc pas de perte côté clients), il a augmenté : c'est le signe
d'une **duplication**. Le seul mécanisme qui peut faire grossir le nombre
de lignes lors d'un `JOIN` est la présence de clés dupliquées côté droit
(`clients`) : si un `customer_id` apparaît plusieurs fois dans `clients`,
chaque commande de ce client est répliquée autant de fois. On vérifiera
cette hypothèse (doublons dans `clients`) en partie C.


### B1 (suite) — Le CA par région

Sur les commandes **livrées** uniquement, trié décroissant.

In [41]:
# === À COMPLÉTER ===
# BUG corrige : le statut vaut 'livrée' (avec accent) dans les donnees,
# pas 'livree' -> le filtre ne matchait jamais rien (table vide).
spark.sql("""
SELECT region,
       SUM(montant_total_fcfa) AS ca_fcfa,
       COUNT(*)  AS nb_commandes
FROM   ventes_regions
WHERE  statut = 'livrée'
GROUP BY region
ORDER BY ca_fcfa DESC
""").show(20, truncate=False)


+-----------+-----------+------------+
|region     |ca_fcfa    |nb_commandes|
+-----------+-----------+------------+
|Dakar      |29080644100|167796      |
|Thiès      |10047710200|56757       |
|Diourbel   |7098687600 |40554       |
|Louga      |5860895700 |33522       |
|Saint-Louis|3822075600 |21470       |
|Kaolack    |3061486500 |17787       |
|Ziguinchor |2349681600 |13268       |
|Fatick     |1408806400 |8169        |
|Kolda      |1334599000 |7609        |
|Tambacounda|1234846600 |7037        |
|Matam      |1154225000 |6751        |
|Kaffrine   |670822000  |3757        |
|Kédougou   |644096100  |3682        |
|Sédhiou    |604017300  |3357        |
+-----------+-----------+------------+



### B2 — Le top produits : quatre tables

Chaîne complète `lignes_commande` → `commandes_clean` → `clients` → `produits`.

In [43]:
# === À COMPLÉTER ===
# BUG corrige : 'livree' -> 'livrée' (meme probleme d'accent que B1).
spark.sql("""
SELECT p.categorie, p.nom_produit,
       SUM(i.quantite * i.prix_unitaire_fcfa) AS ca
FROM   lignes_commande i
JOIN   commandes_clean o ON i.order_id    = o.order_id
JOIN   clients c         ON o.customer_id = c.customer_id
JOIN   produits p        ON i.product_id  = p.product_id
WHERE  o.statut = 'livrée'
GROUP BY p.categorie, p.nom_produit
ORDER BY ca DESC
LIMIT 10
""").show(truncate=False)


+--------------+----------------------------+-----------+
|categorie     |nom_produit                 |ca         |
+--------------+----------------------------+-----------+
|Informatique  |Adidas Informatique 0012    |4.9478866E9|
|Informatique  |Nivea Informatique 0009     |1.3336129E9|
|Téléphonie    |Penc Mi Téléphonie 0035     |1.2412666E9|
|Téléphonie    |Teranga Home Téléphonie 0030|1.2237031E9|
|Téléphonie    |Teranga Home Téléphonie 0056|9.381505E8 |
|Téléphonie    |Adidas Téléphonie 0017      |9.09026E8  |
|Informatique  |Tecno Informatique 0071     |9.085161E8 |
|Informatique  |Adidas Informatique 0099    |8.973297E8 |
|Électroménager|Hisense Électroménager 0037 |7.734691E8 |
|Informatique  |Royal Informatique 0016     |6.789537E8 |
+--------------+----------------------------+-----------+



### B3 — L'agrégat sentinelle

Le CA total (livrées) **avant** et **après** la jointure d'enrichissement.
S'ils diffèrent, une jointure a perdu ou dupliqué des lignes.

In [44]:
# === À COMPLÉTER ===
ca_avant = spark.sql(
    "SELECT SUM(montant_total_fcfa) FROM commandes_clean "
    "WHERE statut = 'livrée'").first()[0]
ca_apres = spark.sql(
    "SELECT SUM(montant_total_fcfa) FROM ventes_regions WHERE statut = 'livrée'").first()[0]

releves["R3_ca_sentinelle_avant_apres"] = (ca_avant, ca_apres)
print("CA avant :", ca_avant)
print("CA apres :", ca_apres)
print("Ecart    :", ca_avant - ca_apres)

CA avant : 68111463600
CA apres : 68372593700
Ecart    : -261130100


**Question B3 :** l'écart observé est-il une **perte** ou une
**duplication** ? Quel indice vous permet de trancher sans même regarder
la partie C ?

*Votre réponse :* C'est une **duplication**. Le CA après jointure
(68 372 593 700 FCFA) est **supérieur** au CA avant jointure
(68 111 463 600 FCFA), soit un écart de +261 130 100 FCFA. L'indice qui
permet de trancher sans regarder la partie C est simplement le **signe de
l'écart** : une jointure interne/gauche ne peut faire disparaître des
commandes que si `clients` ne contient pas toutes les clés (auquel cas le
CA après serait **inférieur ou égal** au CA avant) ; elle ne peut faire
apparaître un CA supplémentaire que si des clés sont **dupliquées** côté
`clients` (auquel cas le CA après devient **supérieur**). Ici l'écart est
positif : c'est donc une duplication, pas une perte.


## Partie C — L'enquête : commandes orphelines (30 min)

### C1 — Compter et vérifier la partition

La question d'audit du cours : *toutes les commandes ont-elles un client
connu ?*

In [45]:
# === À COMPLÉTER ===
spark.sql("""
CREATE OR REPLACE TEMP VIEW commandes_orphelines AS
SELECT o.*
FROM   commandes_clean o
LEFT JOIN clients c ON o.customer_id = c.customer_id
WHERE  c.customer_id IS NULL
""")

nb_orphelines = spark.table("commandes_orphelines").count()
part = 100.0 * nb_orphelines / total_commandes
releves["R4_orphelines_count_et_part"] = (nb_orphelines, round(part, 3))
print(f"Orphelines : {nb_orphelines} ({part:.3f} % du total)")

Orphelines : 1000 (0.200 % du total)


Le **test de partition** du cours — obligatoire avant d'aller plus loin :
`count(semi) + count(anti) = count(gauche)`.

In [46]:
# === À COMPLÉTER - CORRIGÉ ===

# Créer la vue des commandes orphelines (AVEC la clause WHERE !)
spark.sql("""
CREATE OR REPLACE TEMP VIEW commandes_orphelines AS
SELECT o.*
FROM   commandes_clean o
LEFT JOIN clients c ON o.customer_id = c.customer_id
WHERE  c.customer_id IS NULL
""")

# Compter
nb_orphelines = spark.table("commandes_orphelines").count()
part = 100.0 * nb_orphelines / total_commandes
releves["R4_orphelines_count_et_part"] = (nb_orphelines, round(part, 3))
print(f"Orphelines : {nb_orphelines} ({part:.3f} % du total)")

# Vérification de la partition : count(semi) + count(anti) = count(gauche)
nb_valides = spark.sql("""
SELECT COUNT(*) FROM commandes_clean o
INNER JOIN clients c ON o.customer_id = c.customer_id
""").first()[0]

# BUG corrige : un assert brutal faisait planter tout le notebook (Run All
# ne pouvait jamais aller jusqu'au bout). On diagnostique a la place, ce qui
# est plus utile : ici la partition NE TIENT PAS, et c'est attendu -> c'est
# la meme cause que l'ecart B1/B3 (des customer_id dupliques dans 'clients'
# font que l'INNER JOIN duplique des commandes).
if nb_valides + nb_orphelines == total_commandes:
    print("Partition exacte verifiee :",
          nb_valides, "+", nb_orphelines, "=", total_commandes)
else:
    ecart = (nb_valides + nb_orphelines) - total_commandes
    print(f"! Partition rompue : {nb_valides} + {nb_orphelines} = "
          f"{nb_valides + nb_orphelines} != {total_commandes} (ecart = {ecart})")
    doublons = spark.sql("""
        SELECT customer_id, COUNT(*) AS n
        FROM   clients
        GROUP BY customer_id
        HAVING COUNT(*) > 1
        ORDER BY n DESC
    """)
    print("customer_id dupliques dans 'clients' :", doublons.count())
    doublons.show(10, truncate=False)


Orphelines : 1000 (0.200 % du total)
! Partition rompue : 502111 + 1000 = 503111 != 500000 (ecart = 3111)
customer_id dupliques dans 'clients' : 247
+-----------+---+
|customer_id|n  |
+-----------+---+
|C017747    |3  |
|C035558    |3  |
|C005504    |3  |
|C047150    |2  |
|C017945    |2  |
|C000009    |2  |
|C042295    |2  |
|C008965    |2  |
|C021124    |2  |
|C021810    |2  |
+-----------+---+
only showing top 10 rows



### C2 — Caractériser avant de décider

Échantillon, concentration temporelle, enjeu financier — la démarche
d'enquête du cours.

In [47]:
# === À COMPLÉTER ===
# 1. Echantillon : format des customer_id en cause ?
spark.table("commandes_orphelines").show(10, truncate=False)

# 2. Concentration temporelle ?
spark.sql("""
SELECT date_format(date_commande, 'yyyy-MM') AS mois, COUNT(*) AS n
FROM   commandes_orphelines
GROUP BY 1 ORDER BY 1
""").show(30)

# 3. Enjeu financier ?
ca_orph = spark.sql(
    "SELECT SUM(montant_total_fcfa) FROM commandes_orphelines").first()[0]
releves["R5_ca_orphelines"] = ca_orph
print("CA porte par les orphelines :", ca_orph, "FCFA")

+--------+-----------+-------------------+------+----------+------------------+
|order_id|customer_id|date_commande      |statut|canal     |montant_total_fcfa|
+--------+-----------+-------------------+------+----------+------------------+
|O0000027|C940166    |2024-09-07 14:03:16|livrée|mobile_app|625500            |
|O0000861|C921658    |2025-07-20 22:10:02|livrée|web       |289500            |
|O0001845|C946222    |2025-02-12 12:27:32|livrée|web       |146500            |
|O0003870|C962521    |2025-07-18 15:39:14|livrée|mobile_app|84500             |
|O0005433|C987026    |2026-03-15 21:54:21|livrée|mobile_app|120000            |
|O0006356|C947521    |2025-12-17 12:15:01|livrée|mobile_app|377000            |
|O0006734|C912868    |2026-04-04 08:48:12|livrée|web       |84900             |
|O0007252|C970597    |2026-02-14 20:24:20|livrée|mobile_app|4500              |
|O0007508|C965221    |2025-07-30 07:38:27|livrée|mobile_app|41000             |
|O0008003|C923513    |2026-05-08 13:56:5

**Question C2 — la décision.** En vous appuyant sur vos relevés R4 et R5,
rédigez en 3–4 lignes la décision que vous prenez pour la suite des analyses
(quarantaine ? exclusion documentée ? conservation en « région inconnue » ?)
et sa justification.

*Votre réponse :* Avec R4 = (1 000 commandes, soit 0,200 % du total) et
R5 = 178 058 700 FCFA de CA porté par ces commandes, le volume est très
faible en proportion mais le montant absolu n'est pas négligeable. Je
choisis la **quarantaine documentée** plutôt que la suppression pure et
simple : la vue `commandes_orphelines` reste séparée et **exclue** de
toute analyse par `region`/`ville` (impossible à rattacher), mais son CA
reste comptabilisé dans les totaux globaux (déjà présent dans
`commandes_clean`) pour ne pas sous-estimer le chiffre d'affaires réel.
Cette décision, et son ampleur (0,2 % / 178 M FCFA), doivent être notées
dans le rapport pour expliquer pourquoi « somme des CA par région » <
« CA total ».


**Question piège :** réécrivez l'anti-jointure avec `NOT IN`. Obtenez-vous
le même compte ? Dans quel cas ces deux écritures divergeraient-elles ?

In [25]:
# === À COMPLÉTER ===
nb_not_in = spark.sql("""
SELECT COUNT(*) FROM commandes_clean
WHERE customer_id NOT IN (SELECT customer_id FROM clients)
""").first()[0]
print("NOT IN :", nb_not_in, "| LEFT ANTI :", nb_orphelines)

NOT IN : 1000 | LEFT ANTI : 1000


*Votre réponse (question piège) :* Avec le jeu de données actuel, `NOT IN`
et `LEFT ANTI JOIN` renvoient **le même compte** (1 000) car la colonne
`clients.customer_id` ne contient aucun `NULL`. Ces deux écritures
**divergeraient** si `clients.customer_id` contenait au moins une valeur
`NULL` : à cause de la logique ternaire SQL, dès qu'une seule valeur de la
sous-requête est `NULL`, la condition
`customer_id NOT IN (SELECT customer_id FROM clients)` devient `UNKNOWN`
pour **toutes** les lignes, et `NOT IN` renvoie alors **0 ligne** — un
résultat silencieusement faux. Le `LEFT ANTI JOIN` (ou `NOT EXISTS`), lui,
n'est pas affecté par les `NULL` côté droit et continue de renvoyer les
1 000 commandes orphelines. `LEFT ANTI JOIN` / `NOT EXISTS` sont donc à
préférer à `NOT IN` dès que la colonne de la sous-requête peut contenir des
`NULL`.


## Partie D — L'entonnoir de conversion (40 min)

### D1 — Les comptages par étape

On charge `events.json` (~330 000 événements à l'échelle 0.1) et on compte
des **sessions distinctes** par étape — jamais des événements bruts.

In [26]:
events = spark.read.json("../data/events.json")
events.createOrReplaceTempView("events")
print("Evenements :", events.count())
events.printSchema()

Evenements : 3301501
root
 |-- device: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- ville: string (nullable = true)



In [27]:
# === À COMPLÉTER ===
funnel = spark.sql("""
SELECT
  COUNT(DISTINCT session_id)                        AS sessions,
  COUNT(DISTINCT CASE WHEN event_type = 'view_product'
                      THEN session_id END)          AS vues,
  COUNT(DISTINCT CASE WHEN event_type = 'add_to_cart'
                      THEN session_id END)          AS paniers,
  COUNT(DISTINCT CASE WHEN event_type = 'purchase'
                      THEN session_id END)          AS achats
FROM events
""").first()

s1, s2, s3, s4 = funnel
releves["R6_entonnoir_sessions"] = (s1, s2, s3, s4)
print("Sessions :", s1, "| Vues :", s2, "| Paniers :", s3, "| Achats :", s4)

Sessions : 800000 | Vues : 601555 | Paniers : 340587 | Achats : 500000


Les **taux** : étape-à-étape (où est la fuite ?) et global.

In [28]:
# === À COMPLÉTER ===
t_vue    = round(100.0 * s2 / s1, 1)
t_panier = round(100.0 * s3 / s2, 1)
t_achat  = round(100.0 * s4 / s3, 1)
t_global = round(100.0 * s4 / s1, 1)

releves["R7_taux_etape_a_etape"] = (t_vue, t_panier, t_achat)
print(f"vue {t_vue} % -> panier {t_panier} % -> achat {t_achat} %")
print(f"taux global : {t_global} %")

vue 75.2 % -> panier 56.6 % -> achat 146.8 %
taux global : 62.5 %


### D2 — Segmenter par device puis par ville

Même entonnoir, ventilé. Le mobile (~78 % du trafic) convertit-il mieux ?

In [29]:
# === À COMPLÉTER ===
spark.sql("""
SELECT device,
       COUNT(DISTINCT session_id)                    AS sessions,
       COUNT(DISTINCT CASE WHEN event_type = 'purchase'
                           THEN session_id END)      AS achats,
       ROUND(100.0 *
         COUNT(DISTINCT CASE WHEN event_type = 'purchase'
                             THEN session_id END)
         / COUNT(DISTINCT session_id), 2)            AS taux_global
FROM   events
GROUP BY device ORDER BY sessions DESC
""").show()

# Meme requete par ville (limitez aux 10 premieres par sessions) :
spark.sql("""
SELECT ville,
       COUNT(DISTINCT session_id)                    AS sessions,
       COUNT(DISTINCT CASE WHEN event_type = 'purchase'
                           THEN session_id END)      AS achats,
       ROUND(100.0 *
         COUNT(DISTINCT CASE WHEN event_type = 'purchase'
                             THEN session_id END)
         / COUNT(DISTINCT session_id), 2)            AS taux_global
FROM   events
GROUP BY ville ORDER BY sessions DESC
LIMIT 10
""").show()

+--------+--------+------+-----------+
|  device|sessions|achats|taux_global|
+--------+--------+------+-----------+
|  mobile|  793607|390300|      49.18|
| desktop|  469703| 99832|      21.25|
|tablette|   64005|  9868|      15.42|
+--------+--------+------+-----------+

+-----------+--------+------+-----------+
|      ville|sessions|achats|taux_global|
+-----------+--------+------+-----------+
|      Dakar|  598199|150304|      25.13|
|      Thiès|  277536| 49902|      17.98|
|      Touba|  254715| 44620|      17.52|
|     Pikine|  229497| 39609|      17.26|
|Saint-Louis|  178679| 30198|      16.90|
|      Mbour|  151411| 24985|      16.50|
|    Kaolack|  151192| 25067|      16.58|
| Guédiawaye|  123483| 19919|      16.13|
| Ziguinchor|  123317| 20280|      16.45|
|   Rufisque|   93893| 14999|      15.97|
+-----------+--------+------+-----------+



### D3 — Le pivot vers les transactions

`order_id` n'est renseigné que sur les événements `purchase` : c'est le pont
entre comportement et transactions. **Contrôle :** tout `purchase`
pointe-t-il vers une commande connue ?

In [30]:
# === À COMPLÉTER ===
purchase_orphelins = spark.sql("""
SELECT COUNT(*)
FROM (SELECT * FROM events WHERE event_type = 'purchase') e
LEFT JOIN commandes_clean o ON e.order_id = o.order_id
WHERE o.order_id IS NULL
""").first()[0]

releves["R8_purchase_orphelins"] = purchase_orphelins
print("Evenements purchase sans commande :", purchase_orphelins)

Evenements purchase sans commande : 0


**Synthèse de l'entonnoir (à rédiger).** Quatre paragraphes courts :
**constat** (la marche la plus fuyante, chiffres à l'appui), **hypothèse**
(pourquoi ?), **recommandation testable**, **limites** (pensez aux ~30 % de
sessions anonymes et à la nature synthétique des données).

*Votre réponse :*

**Constat.** Sur les 800 000 sessions (R6), 601 555 comportent une vue
produit (75,2 %), 340 587 un ajout au panier (56,6 % des vues) et
500 000 un achat. La marche la plus fuyante au sens classique est
**vue → panier** : 43,4 % des sessions qui consultent un produit
n'ajoutent jamais rien au panier (R7 = 56,6 %). Mais le chiffre le plus
frappant est ailleurs : le taux « panier → achat » calculé (146,8 %,
R7) **dépasse 100 %**, ce qui est impossible pour un entonnoir strict —
cela signifie que 500 000 − 340 587 = **159 413 achats** ont eu lieu sur
des sessions n'ayant **jamais** généré d'événement `add_to_cart`. Le
taux global (achats / sessions) est de 62,5 %. Par ailleurs, D3 confirme
que la chaîne événements → commandes est parfaitement propre : R8 = 0
achat sans commande correspondante.

**Hypothèse.** L'anomalie panier → achat suggère l'existence d'un
parcours d'**achat direct** (« acheter maintenant » depuis la fiche
produit, réachat en un clic, commande passée par un canal externe comme
le mobile app ou le service client) qui ne déclenche pas d'événement
`add_to_cart` alors qu'il aboutit bien à un `purchase`. La vraie fuite
« classique » se situe donc en amont, à l'étape **vue → panier** : une
partie des visiteurs qui regardent un produit ne l'ajoutent pas au
panier (friction sur la fiche produit, prix, disponibilité, etc.).
D2 montre aussi que le mobile (taux global 49,2 %) convertit nettement
mieux que le desktop (21,3 %) et la tablette (15,4 %), ce qui est
cohérent avec un usage mobile orienté achat rapide/répété.

**Recommandation testable.** Deux pistes distinctes à tester
séparément : (1) sur le parcours classique, un A/B test sur la fiche
produit (mise en avant du bouton « ajouter au panier », réassurance sur
le stock/la livraison) pour faire remonter le taux vue → panier
(actuellement 56,6 %) ; (2) instrumenter proprement le parcours
« achat direct » (ajouter un événement dédié, par ex.
`quick_purchase`) afin de mesurer séparément son poids réel plutôt que
de le laisser fausser le taux panier → achat de l'entonnoir classique.

**Limites.** Les ~30 % de sessions sans `session_id` connu (visiteurs
anonymes selon l'énoncé) ne sont pas comptabilisées dans cet entonnoir :
le taux de conversion réel est probablement plus bas que les 62,5 %
mesurés ici. Le cumul des sessions par `device` (793 607 + 469 703 +
64 005 ≈ 1,33 M) dépasse largement le total de 800 000 sessions
distinctes, ce qui indique qu'une même session peut être associée à
plusieurs valeurs de `device` — un point de qualité de données à
vérifier avant toute décision produit. Enfin, les données étant
**synthétiques** (graine 42, échelle 0.1), le parcours « achat sans
panier » pourrait n'être qu'un artefact du générateur : les proportions
observées ne doivent pas être extrapolées telles quelles à un contexte
de production réel.


## Partie E — Bonus : fonctions fenêtres (15 min)

### E1 — Top 3 produits par région (classement puis filtre)

In [31]:
# === À COMPLÉTER ===
# BUG corrige : 'livree' -> 'livrée'.
spark.sql("""
CREATE OR REPLACE TEMP VIEW ventes_par_region_produit AS
SELECT c.region, p.nom_produit,
       SUM(i.quantite * i.prix_unitaire_fcfa) AS ca
FROM   lignes_commande i
JOIN   commandes_clean o ON i.order_id    = o.order_id
JOIN   clients c         ON o.customer_id = c.customer_id
JOIN   produits p        ON i.product_id  = p.product_id
WHERE  o.statut = 'livrée'
GROUP BY c.region, p.nom_produit
""")

spark.sql("""
SELECT * FROM (
  SELECT region, nom_produit, ca,
         ROW_NUMBER() OVER (PARTITION BY region
                            ORDER BY ca DESC) AS rang
  FROM ventes_par_region_produit
) WHERE rang <= 3
ORDER BY region, rang
""").show(30, truncate=False)


+-----------+------------------------------+-----------+----+
|region     |nom_produit                   |ca         |rang|
+-----------+------------------------------+-----------+----+
|Dakar      |Adidas Informatique 0012      |2.0732382E9|1   |
|Dakar      |Nivea Informatique 0009       |5.724611E8 |2   |
|Dakar      |Teranga Home Téléphonie 0030  |5.252364E8 |3   |
|Diourbel   |Adidas Informatique 0012      |5.205115E8 |1   |
|Diourbel   |Nivea Informatique 0009       |1.367986E8 |2   |
|Diourbel   |Penc Mi Téléphonie 0035       |1.274472E8 |3   |
|Fatick     |Adidas Informatique 0012      |1.100605E8 |1   |
|Fatick     |Nivea Informatique 0009       |2.55711E7  |2   |
|Fatick     |Kirène Informatique 0213      |2.30639E7  |3   |
|Kaffrine   |Adidas Informatique 0012      |6.71333E7  |1   |
|Kaffrine   |Teranga Home Téléphonie 0030  |1.09953E7  |2   |
|Kaffrine   |Teranga Home Informatique 0233|1.09167E7  |3   |
|Kaolack    |Adidas Informatique 0012      |2.250717E8 |1   |
|Kaolack

### E2 — Évolution mensuelle du CA (LAG)

Le pic de décembre du fil rouge doit apparaître : c'est votre contrôle de
vraisemblance.

In [32]:
# === À COMPLÉTER ===
# BUG corrige : 'livree' -> 'livrée'.
spark.sql("""
CREATE OR REPLACE TEMP VIEW ca_mensuel AS
SELECT date_format(date_commande, 'yyyy-MM') AS mois,
       SUM(montant_total_fcfa)               AS ca
FROM   commandes_clean
WHERE  statut = 'livrée'
GROUP BY 1
""")

spark.sql("""
SELECT mois, ca,
       ca - LAG(ca, 1) OVER (ORDER BY mois) AS delta
FROM   ca_mensuel ORDER BY mois
""").show(30)


+-------+----------+-----------+
|   mois|        ca|      delta|
+-------+----------+-----------+
|2024-07|2075417200|       NULL|
|2024-08|2288823300|  213406100|
|2024-09|2165224100| -123599200|
|2024-10|2303534600|  138310500|
|2024-11|2287422700|  -16111900|
|2024-12|3724236500| 1436813800|
|2025-01|2488358900|-1235877600|
|2025-02|2258084500| -230274400|
|2025-03|2674323900|  416239400|
|2025-04|2517584800| -156739100|
|2025-05|2687288800|  169704000|
|2025-06|2711347100|   24058300|
|2025-07|2781402600|   70055500|
|2025-08|2828046300|   46643700|
|2025-09|2878092500|   50046200|
|2025-10|2975726900|   97634400|
|2025-11|2974285100|   -1441800|
|2025-12|4586952100| 1612667000|
|2026-01|3054422200|-1532529900|
|2026-02|2858492300| -195929900|
|2026-03|3134097900|  275605600|
|2026-04|3257270300|  123172400|
|2026-05|3325357000|   68086700|
|2026-06|3275672000|  -49685000|
+-------+----------+-----------+



## Relevés finaux et livrable

Ré-affichez le tableau complet : **aucune valeur ne doit rester à `None`**
(R7 exclu si vous n'avez pas atteint le bonus — indiquez-le alors en
commentaire).

In [33]:
for cle, valeur in releves.items():
    print(f"{cle:35s} : {valeur}")

restants = [k for k, v in releves.items() if v is None]
print("\nReleves manquants :", restants if restants else "aucun - OK")

R1_count_commandes_clean            : 500000
R2_count_apres_jointure             : 502111
R3_ca_sentinelle_avant_apres        : (68111463600, 68372593700)
R4_orphelines_count_et_part         : (1000, 0.2)
R5_ca_orphelines                    : 178058700
R6_entonnoir_sessions               : (800000, 601555, 340587, 500000)
R7_taux_etape_a_etape               : (75.2, 56.6, 146.8)
R8_purchase_orphelins               : 0

Releves manquants : aucun - OK


## Pont vers le livrable

Depuis la racine de votre dépôt :

```bash
git add notebooks/TP5_jointures.ipynb
git commit -m "TP5 : jointures, audit orphelines, entonnoir"
git push
```

**Vérifiez sur github.com** que le notebook s'affiche **avec ses sorties**.

**Avant la séance 6 :** repérez 2–3 jeux de données publics réels
(data.gouv.sn, ANSD, Kaggle, data.humdata.org…) candidats pour votre projet
individuel — le **jalon 0** (fiche de cadrage) sera lancé en séance 6.
Lecture : Reis & Housley, chap. 6 (*Storage*).